In [1]:
# !pip install torch-geometric --no-deps --quiet
!pip install torch-geometric==2.6.1 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.7 MB/s eta 0:00:00


In [2]:
import pandas as pd

df_plus = pd.read_csv('/kaggle/input/datasets/nargeskhani/chexpert-plus/df_chexpert_plus_240401.csv')
print(df_plus.shape)
print(df_plus.columns.tolist())
df_plus.head


(223462, 27)
['path_to_image', 'path_to_dcm', 'frontal_lateral', 'ap_pa', 'deid_patient_id', 'patient_report_date_order', 'report', 'section_narrative', 'section_clinical_history', 'section_history', 'section_comparison', 'section_technique', 'section_procedure_comments', 'section_findings', 'section_impression', 'section_end_of_impression', 'section_summary', 'section_accession_number', 'age', 'sex', 'race', 'ethnicity', 'interpreter_needed', 'insurance_type', 'recent_bmi', 'deceased', 'split']


<bound method NDFrame.head of                                       path_to_image  \
0       train/patient42142/study5/view1_frontal.jpg   
1       train/patient42142/study8/view1_frontal.jpg   
2       train/patient42142/study2/view1_frontal.jpg   
3       train/patient42142/study4/view1_frontal.jpg   
4       train/patient42142/study3/view1_frontal.jpg   
...                                             ...   
223457  train/patient59696/study1/view1_frontal.jpg   
223458  train/patient24732/study1/view1_frontal.jpg   
223459  train/patient12591/study1/view1_frontal.jpg   
223460  train/patient37553/study1/view1_frontal.jpg   
223461  train/patient48017/study1/view1_frontal.jpg   

                                        path_to_dcm frontal_lateral ap_pa  \
0       train/patient42142/study5/view1_frontal.dcm         Frontal    AP   
1       train/patient42142/study8/view1_frontal.dcm         Frontal    AP   
2       train/patient42142/study2/view1_frontal.dcm         Frontal    AP   
3

In [3]:
# ==============================================================
# بخش ۱: Importها (ترکیب هر دو فایل)
# ==============================================================
import os
import re
import time
import json
import pickle
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.data import Data, Batch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image


from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, average_precision_score,
    matthews_corrcoef, cohen_kappa_score, log_loss, brier_score_loss
)

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


# ==============================================================
# بخش ۲: لیست بیماری‌ها
# ==============================================================
DISEASES = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly',
    'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
    'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion',
    'Pleural Other', 'Fracture', 'Support Devices'
]


Device: cuda


In [4]:
for root, dirs, files in os.walk('/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/checkpoint_frozen_continued.pth
/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/best_frozen_continued.pth
/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/checkpoint_upload/best_model.pth
/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/checkpoint_upload/checkpoint.pth


In [5]:

print("="*60)
print("🔍 Checking checkpoint_cnngnn.pth content")
print("="*60)

# مسیر دقیق فایل
checkpoint_path = '/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/checkpoint_cnngnn.pth'

if os.path.exists(checkpoint_path):
    print(f"✅ File found at: {checkpoint_path}")
    size = os.path.getsize(checkpoint_path) / (1024*1024)
    print(f"   Size: {size:.2f} MB")
    
    try:
        ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        print(f"\n📋 Checkpoint Contents:")
        print(f"   Keys: {list(ckpt.keys())}")
        print(f"   Epoch: {ckpt.get('epoch', 'N/A')}")
        print(f"   Best AUC: {ckpt.get('best_auc', 'N/A')}")
        
        if 'history' in ckpt and ckpt['history']:
            hist = ckpt['history']
            if 'val_auc' in hist and hist['val_auc']:
                print(f"   Last AUC: {hist['val_auc'][-1]:.4f}")
                print(f"   Total epochs: {len(hist['val_auc'])}")
            else:
                print("   ⚠️ History is empty")
        else:
            print("   ⚠️ No history found")
            
    except Exception as e:
        print(f"   ❌ Error loading: {e}")
else:
    print(f"❌ File not found at: {checkpoint_path}")

🔍 Checking checkpoint_cnngnn.pth content
❌ File not found at: /kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/checkpoint_cnngnn.pth


In [6]:
print("="*60)
print("🔍 Checking checkpoint-cnngnn-pth dataset")
print("="*60)

# مسیر درست
checkpoint_dir = '/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/'
if os.path.exists(checkpoint_dir):
    print(f"✅ Dataset found at: {checkpoint_dir}")
    
    # لیست فایل‌های داخل دیتاست
    files = os.listdir(checkpoint_dir)
    print(f"   Files in dataset: {files}")
    
    # بررسی فایل‌های .pth
    pth_files = [f for f in files if f.endswith('.pth')]
    if pth_files:
        for f in pth_files:
            full_path = os.path.join(checkpoint_dir, f)
            size = os.path.getsize(full_path) / (1024*1024)
            print(f"\n   ✅ Checkpoint file: {f} ({size:.2f} MB)")
            
            # بارگذاری و بررسی محتوا
            try:
                ckpt = torch.load(full_path, map_location='cpu', weights_only=False)
                print(f"      Epoch: {ckpt.get('epoch', 'N/A')}")
                print(f"      Best AUC: {ckpt.get('best_auc', 'N/A')}")
                
                if 'history' in ckpt and ckpt['history']:
                    hist = ckpt['history']
                    if 'val_auc' in hist and hist['val_auc']:
                        print(f"      Last AUC: {hist['val_auc'][-1]:.4f}")
                        print(f"      Total epochs: {len(hist['val_auc'])}")
                    else:
                        print("      ⚠️ History is empty")
                else:
                    print("      ⚠️ No history found")
                    
            except Exception as e:
                print(f"      ❌ Error loading: {e}")
    else:
        print("   ❌ No .pth files found in this dataset (it might be empty)")
else:
    print(f"❌ Dataset not found at: {checkpoint_dir}")

🔍 Checking checkpoint-cnngnn-pth dataset
✅ Dataset found at: /kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/
   Files in dataset: ['checkpoint_frozen_continued.pth', 'best_frozen_continued.pth', 'checkpoint_upload']

   ✅ Checkpoint file: checkpoint_frozen_continued.pth (42.58 MB)
      Epoch: 14
      Best AUC: 0.8473468294940983
      Last AUC: 0.8030
      Total epochs: 15

   ✅ Checkpoint file: best_frozen_continued.pth (32.27 MB)
      Epoch: N/A
      Best AUC: N/A
      ⚠️ No history found


In [7]:
# ==============================================================
# کلاس GATModule (دو لایه GAT + Global Mean Pooling)
# ==============================================================
class GATModule(nn.Module):
    def __init__(self, in_channels=1024, hidden_channels=128,
                 out_channels=256, heads=8, dropout=0.3):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden_channels, heads=heads,
                             dropout=dropout, concat=True)
        self.gat2 = GATConv(hidden_channels * heads, out_channels, heads=1,
                             dropout=dropout, concat=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = self.dropout(x)

        x = self.gat2(x, edge_index)
        x = F.elu(x)

        graph_repr = global_mean_pool(x, batch)  # [B, out_channels]
        return graph_repr

# ==============================================================
# کلاس مدل نهایی CNN-GNN (DenseNet + GAT)
# نکته مهم: گراف همیشه به‌صورت on-the-fly از فیچرمپ واقعی DenseNet
# ساخته می‌شود؛ دیگر گراف بیرونی/ساختگی گرفته نمی‌شود.
# ==============================================================
class CNNGNNModel(nn.Module):
    def __init__(self, densenet_features, num_classes=14,
                 gat_hidden=128, gat_heads=8, dropout=0.3):
        super().__init__()
        self.cnn_features = densenet_features

        self.gat = GATModule(
            in_channels=1024,
            hidden_channels=gat_hidden,
            out_channels=256,
            heads=gat_heads,
            dropout=dropout
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x, freeze_backbone=True, graph_threshold=0.3):
        """
        Args:
            x: تصاویر ورودی [batch_size, 3, 224, 224]
            freeze_backbone: آیا DenseNet فریز باشد
            graph_threshold: آستانه‌ی ساخت یال در گراف

        گراف همیشه همین‌جا از روی فیچرمپ واقعی DenseNet ساخته می‌شود
        تا اطلاعات تصویر واقعاً به GAT برسد (بدون پیش‌محاسبه یا کش).
        """
        if freeze_backbone:
            with torch.no_grad():
                features = self.cnn_features(x)
        else:
            features = self.cnn_features(x)

        graphs = self.build_graph_from_feature_map(
            features.detach() if freeze_backbone else features,
            threshold=graph_threshold
        )
        graphs = Batch.from_data_list(graphs).to(x.device)

        graph_repr = self.gat(graphs.x, graphs.edge_index, graphs.batch)
        logits = self.classifier(graph_repr)

        return logits, features

    def build_graph_from_feature_map(self, feature_map, threshold=0.3):
        """ساخت گراف از فیچرمپ واقعی DenseNet (شباهت کسینوسی + مجاورت مکانی)"""
        B, C, H, W = feature_map.shape
        num_nodes = H * W

        nodes = feature_map.permute(0, 2, 3, 1).reshape(B, num_nodes, C)

        graphs = []
        for b in range(B):
            node_features = nodes[b]

            norm_features = F.normalize(node_features, p=2, dim=1)
            cosine_sim = torch.mm(norm_features, norm_features.t())

            positions = []
            for i in range(H):
                for j in range(W):
                    positions.append([i / H, j / W])
            positions = torch.tensor(positions, dtype=torch.float, device=feature_map.device)

            diff = positions.unsqueeze(0) - positions.unsqueeze(1)
            dist = torch.sqrt((diff ** 2).sum(dim=2))
            sigma = 1.0
            spatial_prox = torch.exp(-dist ** 2 / (2 * sigma ** 2))

            edge_weight = (0.5 * spatial_prox + 0.5 * cosine_sim)
            mask = edge_weight > threshold
            mask.fill_diagonal_(False)

            edge_index = mask.nonzero(as_tuple=False).t()
            weights = edge_weight[mask]

            graph = Data(x=node_features, edge_index=edge_index, edge_attr=weights)
            graphs.append(graph)

        return graphs
# ==============================================================
# تابع get_base_model (برای DataParallel)
# ==============================================================
def get_base_model(model):
    if isinstance(model, nn.DataParallel):
        return model.module
    return model

print("✅ All classes defined successfully!")

✅ All classes defined successfully!


In [8]:
import torch
import torchvision.models as tv_models
import os

# ==============================================================
# ۱. مشخص کردن مسیر درست چک‌پوینت
# ==============================================================

# اولویت ۱: اگر در working موجود است
checkpoint_path = '/kaggle/working/checkpoint.pth'

# اولویت ۲: از دیتاست آپلود شده
if not os.path.exists(checkpoint_path):
    checkpoint_path = '/kaggle/input/datasets/nargeskhani/dateset-epoch10/checkpoint.pth'

print(f"📂 Loading checkpoint from: {checkpoint_path}")
print(f"  exists: {os.path.exists(checkpoint_path)}")

# ==============================================================
# ۲. بارگذاری چک‌پوینت
# ==============================================================

if os.path.exists(checkpoint_path):
    old_ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    old_state = old_ckpt['model_state_dict']
    print("✅ Checkpoint loaded successfully!")
else:
    raise FileNotFoundError(f"Checkpoint not found at {checkpoint_path}")

# ==============================================================
# ۳. استخراج وزن‌های DenseNet
# ==============================================================

print("🔍 Extracting DenseNet weights...")
backbone_state = {}
for key, value in old_state.items():
    if key.startswith('features.'):
        new_key = key.replace('features.', '', 1)
        backbone_state[new_key] = value

print(f"✅ Extracted {len(backbone_state)} layers")

# ==============================================================
# ۴. ساخت DenseNet جدید با وزن‌های انتقالی
# ==============================================================

# ساخت DenseNet جدید
densenet_new = tv_models.densenet121(weights=None)
# densenet_for_graph = densenet_new.features  # فقط قسمت ویژگی‌ها
# densenet_for_graph = densenet_for_graph.to('cpu')  # روی CPU برای ساخت گراف
# بارگذاری وزن‌های انتقالی از checkpoint قدیمی
missing, unexpected = densenet_new.features.load_state_dict(backbone_state, strict=False)

print(f"  Missing keys: {len(missing)}")
print(f"  Unexpected keys: {len(unexpected)}")

if len(missing) > 0:
    print(f"  ⚠️ Warning: {len(missing)} missing keys (first 3): {missing[:3]}")

# ==============================================================
# ۵. ساخت مدل نهایی
# ==============================================================

print("🏗️ Building CNNGNNModel...")
model = CNNGNNModel(
    densenet_features=densenet_new.features,
    num_classes=14,
    gat_hidden=128,
    gat_heads=8,
    dropout=0.3
)

print("✅ Warm-start completed!")
print("📌 DenseNet weights transferred from old checkpoint.")
print("📌 GAT and Classifier are randomly initialized.")

📂 Loading checkpoint from: /kaggle/input/datasets/nargeskhani/dateset-epoch10/checkpoint.pth
  exists: True
✅ Checkpoint loaded successfully!
🔍 Extracting DenseNet weights...
✅ Extracted 725 layers
  Missing keys: 0
  Unexpected keys: 0
🏗️ Building CNNGNNModel...
✅ Warm-start completed!
📌 DenseNet weights transferred from old checkpoint.
📌 GAT and Classifier are randomly initialized.


In [9]:
import torch

best_checkpoint_path = '/kaggle/input/datasets/nargeskhani/checkpoint-cnngnn-pth/best_frozen_continued.pth'
state_dict = torch.load(best_checkpoint_path, map_location='cpu', weights_only=False)
get_base_model(model).load_state_dict(state_dict)
print("✅ وزن‌های AUC=0.8473 بارگذاری شد.")

✅ وزن‌های AUC=0.8473 بارگذاری شد.


In [10]:
# ==============================================================
# بخش ۴: CheXpertDataset (ترکیب: مدیریت خطا از docx + fix_path از notebook)
# ==============================================================
class CheXpertDataset(Dataset):
    def __init__(self, csv_path, img_root, transform=None,
                 uncertain_label='smoothing', verbose=True):
        df = pd.read_csv(csv_path)

        def fix_path(path):
            if path.startswith('CheXpert-v1.0-small/'):
                return path.replace('CheXpert-v1.0-small/', '')
            return path
        df['Path'] = df['Path'].apply(fix_path)

        # نکته‌ی مهم برای حافظه: دیگر DataFrame پانداس را نگه نمی‌داریم و
        # __getitem__ دیگر .iloc[] صدا نمی‌زند. فقط آرایه‌های numpy ساده
        # نگه داشته می‌شوند تا با DataLoaderِ چندworkerای (fork) نشتی
        # حافظه‌ی معروف Copy-on-Write مربوط به pandas رخ ندهد.
        self.paths = df['Path'].values
        self.labels_raw = df[DISEASES].values.astype('float32')
        self.length = len(df)

        self.img_root = img_root
        self.transform = transform
        self.uncertain_label = uncertain_label
        self.verbose = verbose

        if verbose:
            print(f"Loading dataset from: {csv_path}")
            print(f"Total rows: {self.length}")
            full_path = os.path.join(img_root, self.paths[0])
            print(f"Sample path exists: {os.path.exists(full_path)}")

    def process_labels(self, raw_row):
        labels = []
        for val in raw_row:
            if val == 1.0:
                labels.append(1.0)
            elif val == 0.0:
                labels.append(0.0)
            elif val == -1.0:
                if self.uncertain_label == 'smoothing':
                    labels.append(0.55)
                elif self.uncertain_label == 'ignore':
                    labels.append(-1.0)
                else:
                    labels.append(1.0)
            else:
                labels.append(0.0)
        return torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_root, self.paths[idx])

        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            if self.verbose:
                print(f"Warning: could not load {img_path}, using placeholder")
            image = Image.new('RGB', (224, 224), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        labels = self.process_labels(self.labels_raw[idx])
        return image, labels

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==============================================================
# DataLoader
# (دیگر نیازی به collate_fn سفارشی برای گراف نیست، چون دیتاست
#  دیگر گراف تولید نمی‌کند — مدل خودش در forward گراف را از فیچرمپ
#  واقعی می‌سازد. collate پیش‌فرض PyTorch برای (image, labels) کافی‌ست)
# ==============================================================
def create_dataloaders(train_dataset, val_dataset, batch_size=16,
                       num_workers=0, device=None):
    if device is None:
        device = DEVICE

    loader_kwargs = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": device.type == "cuda",
    }

    if num_workers > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 2

    train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)

    return train_loader, val_loader

In [11]:
# ==============================================================
# Loss سفارشی
# ==============================================================
class MaskedBCEWithLogitsLoss(nn.Module):
    def forward(self, logits, target):
        mask = target >= 0
        logits = logits[mask]
        target = target[mask].float()
        if logits.numel() == 0:
            return logits.sum() * 0.0
        return F.binary_cross_entropy_with_logits(logits, target, reduction="mean")



# ==============================================================
# آموزش یک Epoch با پشتیبانی از گراف
# ==============================================================
def train_epoch(model, loader, optimizer, criterion, device, scaler,
                freeze_backbone=True, use_amp=True,
                epoch=None, checkpoint_path=None, save_every_n_batches=300,
                best_auc=0.0, history=None):
    import psutil
    import gc
    model.train()
    total_loss = 0.0
    
    progress = tqdm(loader, desc="  Training", leave=False, ncols=100)
    
    for batch_idx, (images, labels) in enumerate(progress):
        images = images.to(device, non_blocking=(device.type == "cuda"))
        labels = labels.to(device, non_blocking=(device.type == "cuda"))
    
        optimizer.zero_grad(set_to_none=True)
    
        with torch.autocast(device_type=device.type, dtype=torch.float16,
                           enabled=use_amp and (device.type == "cuda")):
            logits, _ = model(images, freeze_backbone=freeze_backbone)
            loss = criterion(logits, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        progress.set_postfix({'Loss': f'{loss.item():.4f}'})
        
        # پایش حافظه
        if batch_idx % 500 == 0:
            process = psutil.Process(os.getpid())
            ram_mb = process.memory_info().rss / (1024 * 1024)
            print(f"🧠 Batch {batch_idx} | RAM: {ram_mb:.1f} MB")
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
        
        if batch_idx % 100 == 0:
            gc.collect()
        
        # Checkpoint میان‌epochی
        if (checkpoint_path and epoch is not None and batch_idx > 0
                and batch_idx % save_every_n_batches == 0):
            torch.save({
                'epoch': epoch,
                'mid_epoch': True,
                'model_state_dict': get_base_model(model).state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_auc': best_auc,
                'history': history or {},
            }, checkpoint_path)
    
    return total_loss / max(len(loader), 1)
    
# ==============================================================
# اعتبارسنجی با پشتیبانی از گراف
# ==============================================================
def validate(model, loader, criterion, device, use_amp=False, freeze_backbone=True):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    
    progress = tqdm(loader, desc="  Validating", leave=False, ncols=100)
    
    with torch.inference_mode():
        for images, labels in progress:
            images = images.to(device, non_blocking=(device.type == "cuda"))
            labels = labels.to(device, non_blocking=(device.type == "cuda"))

            with torch.autocast(device_type=device.type, dtype=torch.float16,
                               enabled=use_amp and (device.type == "cuda")):
                logits, _ = model(images, freeze_backbone=freeze_backbone)
                loss = criterion(logits, labels)
            
            predictions = torch.sigmoid(logits)
            total_loss += loss.item()
            all_preds.append(predictions.float().cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            progress.set_postfix({'Loss': f'{loss.item():.4f}'})
    
    all_preds = np.concatenate(all_preds, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    
    aucs, aucs_by_disease = [], {}
    for i, disease in enumerate(DISEASES):
        mask = all_labels[:, i] >= 0
        if mask.sum() > 0 and len(np.unique(all_labels[mask, i])) > 1:
            auc = roc_auc_score(all_labels[mask, i], all_preds[mask, i])
            aucs.append(auc)
            aucs_by_disease[disease] = auc
    
    mean_auc = float(np.mean(aucs)) if aucs else 0.0
    avg_loss = total_loss / max(len(loader), 1)
    return avg_loss, mean_auc, aucs_by_disease, all_preds, all_labels

In [12]:
# ==============================================================
# بخش ۱۴: حلقه‌ی اصلی آموزش (Orchestration)
# ترکیب: Scheduler + Early Stopping + DataParallel (از docx)
#         با Checkpoint/Resume مقاوم + آپلود Kaggle (از notebook خودمون)
# ==============================================================
def train_model(model, train_loader, val_loader,
                 num_epochs=30, stage1_epochs=10,
                 checkpoint_path='/kaggle/working/checkpoint.pth',
                 best_model_path='/kaggle/working/best_model.pth',
                 resume=True, use_amp=True, patience=7,
                 lr_stage1=1e-3, lr_stage2=1e-5,
                 kaggle_dataset_handle=None, device=None):
    """
    استراتژی دو-مرحله‌ای:
      مرحله ۱ (epoch 0..stage1_epochs-1): DenseNet فریز، فقط GAT+classifier train میشن
      مرحله ۲ (بعدش): DenseNet هم unfreeze میشه، با LR کمتر (fine-tune مشترک)
    """
    if device is None:
        device = DEVICE
    model = model.to(device)
    print(f"در حال اجرا روی تک‌GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    # if torch.cuda.device_count() > 1:
    #     print(f"استفاده از {torch.cuda.device_count()} GPU با DataParallel")
    #     model = nn.DataParallel(model)

    criterion = MaskedBCEWithLogitsLoss()
    scaler = torch.amp.GradScaler(enabled=use_amp and device.type == "cuda")

    start_epoch = 0
    best_auc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': []}
    epochs_without_improvement = 0

    base = get_base_model(model)
    optimizer = optim.Adam(
        list(base.gat.parameters()) + list(base.classifier.parameters()),
        lr=lr_stage1
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3
    )

    # ---- Resume از checkpoint (اگه بود) ----
    if resume and os.path.exists(checkpoint_path):
        print(f"Checkpoint پیدا شد -- بارگذاری از {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
        is_mid_epoch = ckpt.get('mid_epoch', False)
        start_epoch = ckpt['epoch'] if is_mid_epoch else ckpt['epoch'] + 1
        best_auc = ckpt.get('best_auc', 0.0)
        history = ckpt.get('history', history)

        if start_epoch >= stage1_epochs:
            optimizer = optim.Adam(base.parameters(), lr=lr_stage2)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=3
            )

        get_base_model(model).load_state_dict(ckpt['model_state_dict'])
        try:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        except Exception as e:
            print(f"هشدار: optimizer_state_dict لود نشد (مشکلی ایجاد نمی‌کند): {e}")

        print(f"ادامه از epoch {start_epoch + 1} | بهترین AUC تا الان: {best_auc:.4f}")
    else:
        print("Checkpoint ای پیدا نشد -- آموزش از ابتدا شروع می‌شه")

    # ---- حلقه‌ی اصلی ----
    for epoch in range(start_epoch, num_epochs):
        stage2_active = epoch >= stage1_epochs
        freeze_backbone = not stage2_active

        if epoch == stage1_epochs:
            print("\n>>> ورود به مرحله ۲: Unfreeze کردن DenseNet برای fine-tune مشترک")
            optimizer = optim.Adam(base.parameters(), lr=lr_stage2)
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=3
            )

        print(f"\nEpoch {epoch + 1}/{num_epochs} "
              f"[{'Stage 1: Frozen' if freeze_backbone else 'Stage 2: Fine-tune'}]")

        train_loss = train_epoch(
            model, train_loader, optimizer, criterion, device, scaler,
            freeze_backbone=freeze_backbone, use_amp=use_amp,
            epoch=epoch, checkpoint_path=checkpoint_path,
            best_auc=best_auc, history=history
        )
        val_loss, val_auc, aucs_by_disease, _, _ = validate(
            model, val_loader, criterion, device, use_amp=use_amp,
            freeze_backbone=freeze_backbone
        )

        scheduler.step(val_auc)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auc'].append(val_auc)

        print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val AUC: {val_auc:.4f} | LR: {current_lr:.2e}")

        improved = val_auc > best_auc
        if improved:
            best_auc = val_auc
            epochs_without_improvement = 0
            torch.save(get_base_model(model).state_dict(), best_model_path)
            print(f"  New best AUC: {best_auc:.4f} -- مدل ذخیره شد")
        else:
            epochs_without_improvement += 1

        # ---- Checkpoint پایان-epoch ----
        torch.save({
            'epoch': epoch,
            'mid_epoch': False,
            'model_state_dict': get_base_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_auc': best_auc,
            'history': history,
        }, checkpoint_path)

        # ---- آپلود اختیاری به Kaggle Dataset ----
        if kaggle_dataset_handle:
            try:
                import kagglehub
                upload_dir = '/kaggle/working/checkpoint_upload'
                os.makedirs(upload_dir, exist_ok=True)
                if os.path.exists(checkpoint_path):
                    shutil.copy(checkpoint_path, os.path.join(upload_dir, 'checkpoint.pth'))
                if os.path.exists(best_model_path):
                    shutil.copy(best_model_path, os.path.join(upload_dir, 'best_model.pth'))
                kagglehub.dataset_upload(
                    kaggle_dataset_handle, upload_dir,
                    version_notes=f'checkpoint after epoch {epoch + 1}'
                )
                print("  Checkpoint به Kaggle Dataset آپلود شد")
            except Exception as e:
                # این آپلود هیچ‌وقت نباید کل training رو متوقف کنه
                print(f"  هشدار: آپلود به Kaggle ناموفق بود (training ادامه پیدا می‌کند): {e}")

        # ---- Early Stopping ----
        if epochs_without_improvement >= patience:
            print(f"\nEarly stopping -- {patience} epoch بدون بهبود")
            break

    print(f"\nآموزش تمام شد. بهترین AUC: {best_auc:.4f}")
    return model, history

In [13]:
# ==============================================================
# آموزش مدل با Weight Transfer و گراف‌های از پیش ساخته
# ==============================================================

# ۱. مسیرها
KAGGLE_PATH = '/kaggle/input/datasets/ashery/chexpert'

# ۲. ساخت دیتاست‌ها با گراف‌های از پیش ساخته
print("\n📂 Loading datasets with precomputed graphs...")

# در سلول آموزش:
train_dataset = CheXpertDataset(
    csv_path=os.path.join(KAGGLE_PATH, 'train.csv'),
    img_root=KAGGLE_PATH,
    transform=train_transform,
    uncertain_label='smoothing',
    verbose=True
)

val_dataset = CheXpertDataset(
    csv_path=os.path.join(KAGGLE_PATH, 'valid.csv'),
    img_root=KAGGLE_PATH,
    transform=val_transform,
    uncertain_label='smoothing',
    verbose=True
)
# ۳. ایجاد DataLoader
train_loader, val_loader = create_dataloaders(
    train_dataset, val_dataset,
    batch_size=10,  # کاهش batch_size برای مدیریت بهتر حافظه
    num_workers=2
)

print(f"✅ Train: {len(train_dataset)} | Validation: {len(val_dataset)}")

# ۴. آموزش
print("\n" + "="*60)
print("🚀 STARTING TRAINING (DenseNet + GAT with Precomputed Graphs)")
print("="*60)


📂 Loading datasets with precomputed graphs...
Loading dataset from: /kaggle/input/datasets/ashery/chexpert/train.csv
Total rows: 223414
Sample path exists: True
Loading dataset from: /kaggle/input/datasets/ashery/chexpert/valid.csv
Total rows: 234
Sample path exists: True
✅ Train: 223414 | Validation: 234

🚀 STARTING TRAINING (DenseNet + GAT with Precomputed Graphs)


In [14]:

# model, history = train_model(
#     model=model,                         # از Warm-start آمده
#     train_loader=train_loader,
#     val_loader=val_loader,
#     num_epochs=15,                       # تعداد کل دوره‌ها
#     stage1_epochs=15,                    # مرحله اول با DenseNet فریز
#     checkpoint_path='/kaggle/working/checkpoint_cnngnn.pth',
#     best_model_path='/kaggle/working/best_cnngnn.pth',
#     resume=False,                       # مهم! چون مدل جدید است
#     use_amp=True,
#     patience=5,
#     lr_stage1=1e-3,
#     lr_stage2=1e-5,
#     kaggle_dataset_handle='nargeskhani/checkpoint-cnngnn-pth',
#     device=DEVICE
# )

# print("✅ Training completed!")

In [15]:
model = model.to(DEVICE)

val_loss, val_auc, _, _, _ = validate(
    model, val_loader, MaskedBCEWithLogitsLoss(), DEVICE,
    use_amp=True, freeze_backbone=True
)
print(f"AUC تأییدی: {val_auc:.4f}")

AUC تأییدی: 0.8473


In [16]:
# ==============================================================
# جستجوی خودکار برای پیدا کردن بهترین Checkpoint
# ==============================================================

import os
import torch

print("="*60)
print("🔍 Searching for the best checkpoint...")
print("="*60)

# لیست مسیرهایی که باید بررسی شوند
checkpoint_paths = [
    '/kaggle/input/datasets/nargeskhani/dateset-epoch10/checkpoint.pth',
    '/kaggle/input/datasets/nargeskhani/chexpert-densenet-baseline/checkpoint.pth',
    '/kaggle/working/checkpoint.pth',
    '/kaggle/working/checkpoint_cnngnn.pth',
]

best_auc = -1.0
best_path = None
best_epoch = -1

for path in checkpoint_paths:
    if os.path.exists(path):
        print(f"\n📂 Checking: {path}")
        print(f"   File exists: ✅")
        
        try:
            # بارگذاری checkpoint
            checkpoint = torch.load(path, map_location='cpu', weights_only=False)
            
            # استخراج اطلاعات
            epoch = checkpoint.get('epoch', 'N/A')
            auc = checkpoint.get('best_auc', -1.0)
            
            # اگر 'best_auc' وجود نداشت، از history استخراج کن
            if auc == -1.0 and 'history' in checkpoint:
                history = checkpoint['history']
                if 'val_auc' in history and len(history['val_auc']) > 0:
                    auc = max(history['val_auc'])
                    epoch = len(history['val_auc']) - 1
            
            # اگر 'best_auc' از داخل state_dict باشد
            if auc == -1.0:
                state_dict = checkpoint.get('model_state_dict', {})
                # معمولاً best_auc در checkpoint ذخیره می‌شود
                pass
            
            print(f"   📊 Epoch: {epoch}")
            print(f"   📈 Best AUC: {auc:.4f}")
            
            # ذخیره بهترین
            if auc > best_auc:
                best_auc = auc
                best_path = path
                best_epoch = epoch
                
        except Exception as e:
            print(f"   ❌ Error loading: {e}")
    else:
        print(f"\n📂 Checking: {path}")
        print(f"   File exists: ❌ (Not found)")

# ==============================================================
# نمایش بهترین نتیجه
# ==============================================================

print("\n" + "="*60)
print("🏆 BEST CHECKPOINT FOUND")
print("="*60)

if best_path is not None:
    print(f"📂 Path: {best_path}")
    print(f"📊 Epoch: {best_epoch}")
    print(f"📈 Best AUC: {best_auc:.4f}")
    
    # بارگذاری کامل checkpoint برای نمایش اطلاعات بیشتر
    try:
        checkpoint = torch.load(best_path, map_location='cpu', weights_only=False)
        print(f"\n📋 Checkpoint keys: {list(checkpoint.keys())}")
        
        if 'history' in checkpoint:
            history = checkpoint['history']
            if 'val_auc' in history:
                print(f"   📈 All AUCs: {[round(x, 4) for x in history['val_auc']]}")
            if 'train_loss' in history:
                print(f"   📉 Train Losses: {[round(x, 4) for x in history['train_loss'][:5]]}...")
    except:
        pass
else:
    print("❌ No checkpoint found in any of the specified paths!")

print("="*60)

🔍 Searching for the best checkpoint...

📂 Checking: /kaggle/input/datasets/nargeskhani/dateset-epoch10/checkpoint.pth
   File exists: ✅
   📊 Epoch: 31
   📈 Best AUC: 0.8504

📂 Checking: /kaggle/input/datasets/nargeskhani/chexpert-densenet-baseline/checkpoint.pth
   File exists: ❌ (Not found)

📂 Checking: /kaggle/working/checkpoint.pth
   File exists: ❌ (Not found)

📂 Checking: /kaggle/working/checkpoint_cnngnn.pth
   File exists: ❌ (Not found)

🏆 BEST CHECKPOINT FOUND
📂 Path: /kaggle/input/datasets/nargeskhani/dateset-epoch10/checkpoint.pth
📊 Epoch: 31
📈 Best AUC: 0.8504

📋 Checkpoint keys: ['epoch', 'mid_epoch', 'model_state_dict', 'optimizer_state_dict', 'best_auc', 'train_losses', 'val_losses', 'val_aucs']
